# Step 0 - gates for the `mobile_2` reference pose

Before building any estimator for the sensor-constrained agent, three questions must be answered
from the data, because a *no* to any of them changes the design:

| Gate | Question | If it fails |
|------|----------|-------------|
| **0a** | where are the surveyed boards in the map frame, and how good is each? | fix the survey first |
| **0b** | how often does `mobile_2` actually see a board? | the fiducial arm collapses to "VSLAM with one anchor" - reframe, don't fuse |
| **0c** | are the boards still where the survey (96 min earlier) says, and is VSLAM scale sane? | board constraints are poisoned - re-survey or drop the moved board |

Cells cache to `WORK/`; re-running reads the cache unless a cell's `FORCE = True`.
The last cell saves the assigned sightings for the next notebook (depth clouds, registration,
pose graph), so nothing here needs to run twice.

In [ ]:
# ============================== CONFIG ==============================
from pathlib import Path

# --- inputs -------------------------------------------------------------
BAG        = Path("/home/wicoms-robot/data/mirc_dataset_coop2_20260828_merged")   # rosbag2 dir (metadata.yaml inside)
SURVEY     = Path("/home/wicoms-robot/workspaces/isaac_ros-dev/map_stages_20260828/board_survey.json")

WORK       = Path("./m2_reference"); WORK.mkdir(exist_ok=True)

# --- topics -------------------------------------------------------------
T_INFRA1     = "/mobile_2/infra1/image_rect_raw"
T_INFRA1_CI  = "/mobile_2/infra1/camera_info"
T_COLOR      = "/mobile_2/color/image_raw"          # fallback if IR speckle kills detection
T_COLOR_CI   = "/mobile_2/color/camera_info"
T_VO         = "/mobile_2/visual_slam/tracking/odometry"

# --- census -------------------------------------------------------------
CENSUS_STRIDE   = 2        # detect every Nth infra1 frame (27.5 Hz -> ~14 Hz)
MIN_CORNERS     = 6        # ChArUco corners needed to accept a sighting
MAX_REPROJ_PX   = 1.0      # reject a sighting above this mean reprojection error
MARKER_RATIO    = 0.75     # marker_len / square_len; not in the survey JSON, detection-only

LIMIT_FRAMES    = None     # e.g. 500 while smoke-testing, None for the full bag
print("config loaded ->", WORK.resolve())

In [ ]:
# ============================== IMPORTS + helpers ==============================
import json, math, time
import numpy as np
from scipy.spatial.transform import Rotation as Rot
import matplotlib.pyplot as plt

def Rt(R, t):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = t; return T

def inv(T):
    R = T[:3, :3]; o = np.eye(4); o[:3, :3] = R.T; o[:3, 3] = -R.T @ T[:3, 3]; return o

def q_to_R(q):        # q = xyzw
    return Rot.from_quat(q).as_matrix()

def apply(T, P):      # P (N,3)
    return np.asarray(P) @ T[:3, :3].T + T[:3, 3]

def interp_traj(ts_src, Ts_src, ts_q):
    """SLERP + linear interpolation onto query stamps; clamps at the ends."""
    from scipy.spatial.transform import Slerp
    ts_q = np.clip(ts_q, ts_src[0], ts_src[-1])
    i = np.clip(np.searchsorted(ts_src, ts_q) - 1, 0, len(ts_src) - 2)
    d = ts_src[i + 1] - ts_src[i]
    a = np.where(d > 0, (ts_q - ts_src[i]) / np.where(d > 0, d, 1), 0.0)
    sl = Slerp(ts_src, Rot.from_matrix(Ts_src[:, :3, :3]))
    out = np.tile(np.eye(4), (len(ts_q), 1, 1))
    out[:, :3, :3] = sl(ts_q).as_matrix()
    out[:, :3, 3] = Ts_src[i, :3, 3] * (1 - a)[:, None] + Ts_src[i + 1, :3, 3] * a[:, None]
    return out

print("numpy", np.__version__)
import cv2; print("opencv", cv2.__version__)

## Step 0a - board survey -> map frame

The survey JSON stores board poses in the **normalised frame `N`** (`T_N_world`, anchor board at
the origin, a 3.11 deg yaw off the map frame). The GLIM trajectory lives in the **map** frame, so
every board pose is pulled back through `inv(T_N_world)`.

Two things this cell also does, both load-bearing:

1. **Assigns per-board sigmas as `max(section std, loop-closure disagreement)`.** The raw
   `std_mm` is optimistic for boards with a thin second section. `anchor_b` reports 2.01 mm std
   but carries `drift_warning: true` and a *significant* 12.94 mm / 2.64 deg loop closure
   (ratio 4.24) - its second section has n=5 views. Weighting all three boards equally is the
   mistake that makes the ablation look good and then fails the hold-out test.
2. **Flags the ID collision.** `anchor` and `anchor_b` are the *same physical design* -
   DICT_4X4_50, 9x7, 20 mm, `id_offset: 0`. Detection cannot tell them apart by marker ID; they
   must be disambiguated by position against a pose prior. Handled in the census cell.

In [ ]:
# ============================== STEP 0a: board survey ==============================
S = json.loads(SURVEY.read_text())
T_N_world = np.array(S["T_N_world"]); T_world_N = inv(T_N_world)

BOARDS = {}
for name, b in S["boards"].items():
    T_N_b = Rt(q_to_R(b["qxyzw"]), np.array(b["xyz"]))
    lc = b.get("loop_closure", {}) or {}
    sig_t = max(b.get("std_mm", 0.0), lc.get("mm", 0.0)) * 1e-3        # metres
    sig_r = math.radians(max(lc.get("deg", 0.0), 0.2))                 # rad, floor at 0.2 deg
    BOARDS[name] = dict(
        name=name,
        T_map_board=T_world_N @ T_N_b,
        squares=tuple(b["squares"]), square_len=b["square_len"],
        dictionary=b["dictionary"], id_offset=b.get("id_offset", 0),
        sigma_t=sig_t, sigma_r=sig_r,
        n_views=b.get("n_views", 0), std_mm=b.get("std_mm", np.nan),
        drift_warning=bool(b.get("drift_warning", False)),
        lc_significant=bool(lc.get("significant", False)),
    )

print(f"{'board':11s} {'x':>8s} {'y':>8s} {'z':>8s}  {'sig_t':>7s} {'sig_R':>7s} {'views':>5s}  flags")
for n, b in BOARDS.items():
    p = b["T_map_board"][:3, 3]
    fl = ",".join([f for f, on in [("DRIFT", b["drift_warning"]),
                                   ("LC-SIG", b["lc_significant"])] if on]) or "-"
    print(f"{n:11s} {p[0]:8.3f} {p[1]:8.3f} {p[2]:8.3f}  "
          f"{b['sigma_t']*1000:6.1f}mm {math.degrees(b['sigma_r']):6.2f}d {b['n_views']:5d}  {fl}")

# ID collisions: same dictionary + same id_offset => indistinguishable by marker id
from collections import defaultdict
grp = defaultdict(list)
for n, b in BOARDS.items(): grp[(b["dictionary"], b["id_offset"], b["squares"])].append(n)
AMBIG = {k: v for k, v in grp.items() if len(v) > 1}
for k, v in AMBIG.items():
    print(f"\n!! ID COLLISION {v} share {k[0]} offset {k[1]} {k[2]} "
          "-> disambiguate by position, not id")

# board-board baselines: free long-baseline control distances for the rangefinder
names = list(BOARDS)
print("\nsurveyed baselines (shoot these with the rangefinder - long, one in a corridor):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        d = np.linalg.norm(BOARDS[names[i]]["T_map_board"][:3, 3] -
                           BOARDS[names[j]]["T_map_board"][:3, 3])
        print(f"  {names[i]:11s} -> {names[j]:11s} {d:7.3f} m")

## Step 0b - sighting census  **(GATE)**

Everything downstream depends on one unknown: **how often does `mobile_2` actually see a board?**

Decision rule after this cell:
- **>= 4 well-separated sighting windows** across the trajectory -> the three-arm ablation is real.
- **1-2 windows, clustered** -> arm B is "VSLAM with one anchor". Say so and reframe the paper
  claim as *anchored vs unanchored VSLAM*, not a fusion ablation.

**Detect in `infra1`, not `color`.** infra1 is global shutter (color is rolling - at 1 m/s a
board is skewed), it is factory-rectified, and depth is already registered to it, so the board
pose lands in the depth frame with no extra extrinsic. The one risk is the IR projector pattern
sitting on the board: watch `mean_reproj_px` against the survey's 0.22-0.39 px. If it is much
worse, flip `USE_COLOR = True` and carry `T_infra1_color`.

In [ ]:
# ============================== bag reader ==============================
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utility import get_message

def bag_reader(path):
    r = rosbag2_py.SequentialReader()
    r.open(rosbag2_py.StorageOptions(uri=str(path), storage_id="mcap"),
           rosbag2_py.ConverterOptions("", ""))
    types = {t.name: t.type for t in r.get_all_topics_and_types()}
    return r, types

def iter_topic(path, topic, stride=1, limit=None):
    """Yield (t_sec, msg). t_sec is the message header stamp when present, else bag time."""
    r, types = bag_reader(path)
    if topic not in types: raise KeyError(f"{topic} not in bag; have {sorted(types)[:8]}...")
    cls = get_message(types[topic])
    f = rosbag2_py.StorageFilter(); f.topics = [topic]; r.set_filter(f)
    i = n = 0
    while r.has_next():
        _, data, t_bag = r.read_next()
        if i % stride: i += 1; continue
        i += 1
        m = deserialize_message(data, cls)
        h = getattr(m, "header", None)
        t = (h.stamp.sec + h.stamp.nanosec * 1e-9) if h is not None else t_bag * 1e-9
        yield t, m
        n += 1
        if limit and n >= limit: break

def first_msg(path, topic):
    for t, m in iter_topic(path, topic, limit=1): return m
    return None

def camera_K(path, topic_ci):
    ci = first_msg(path, topic_ci)
    K = np.array(ci.k).reshape(3, 3)
    D = np.array(ci.d, dtype=float)
    # image_rect_raw / rect_color are already rectified -> D must be treated as zero
    return K, np.zeros(5), ci.width, ci.height

def img_to_np(m):
    a = np.frombuffer(m.data, dtype=np.uint8)
    enc = m.encoding
    if enc in ("mono8", "8UC1"):   return a.reshape(m.height, m.step)[:, :m.width]
    if enc == "16UC1":             return a.view(np.uint16).reshape(m.height, m.step // 2)[:, :m.width]
    ch = {"rgb8": 3, "bgr8": 3, "rgba8": 4, "bgra8": 4}[enc]
    im = a.reshape(m.height, m.step // ch, ch)[:, :m.width]
    return im[..., ::-1][..., :3] if enc.startswith("rgb") else im[..., :3]

def odom_to_T(m):
    p = m.pose.pose.position; o = m.pose.pose.orientation
    return Rt(q_to_R([o.x, o.y, o.z, o.w]), np.array([p.x, p.y, p.z]))

K_IR, _, W_IR, H_IR = camera_K(BAG, T_INFRA1_CI)
print("infra1 K\n", K_IR, f"\n{W_IR}x{H_IR}")

In [ ]:
# ============================== ChArUco detection ==============================
# The board-frame axis convention is NOT fully determined by the survey JSON
# (board_origin="center", board_axes="ros" name a convention without defining its rotation).
# It CANNOT be recovered from detections alone: PnP under any rigid candidate reproduces the
# same image corners and the same board ORIGIN - only the board's orientation frame changes.
# The check cell below picks it from board-to-board RELATIVE rotations against the survey;
# a wrong convention is off by ~90-180 deg there, VSLAM drift by ~1 deg.
AXIS_CANDIDATES = {
    "cv":       np.eye(3),                                          # OpenCV: X right, Y down, Z out
    "xy_flip":  np.diag([1.0, -1.0, -1.0]),                          # X right, Y up,  Z in
    "ros":      np.array([[0., -1., 0.], [0., 0., -1.], [1., 0., 0.]]),  # X out, Y left, Z up
    "ros_180":  np.array([[0., 1., 0.], [0., 0., -1.], [-1., 0., 0.]]),  # ros, yawed 180
}
BOARD_AXES = "cv"   # overwritten by the auto-select cell

def make_board(spec):
    d = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, spec["dictionary"]))
    sx, sy = spec["squares"]; sq = spec["square_len"]; mk = sq * MARKER_RATIO
    try:                                  # OpenCV >= 4.7
        b = cv2.aruco.CharucoBoard((sx, sy), sq, mk, d)
        b.setLegacyPattern(True)
    except AttributeError:                # OpenCV 4.6
        b = cv2.aruco.CharucoBoard_create(sx, sy, sq, mk, d)
    return b, d

def board_object_points(spec, axes=None):
    """Interior ChArUco corners, (sx-1)*(sy-1) x 3, in the SURVEYED board frame."""
    sx, sy = spec["squares"]; sq = spec["square_len"]
    j, i = np.meshgrid(np.arange(1, sy), np.arange(1, sx), indexing="ij")
    P = np.column_stack([i.ravel() * sq, j.ravel() * sq, np.zeros(i.size)])
    P -= np.array([sx * sq / 2, sy * sq / 2, 0.0])           # board_origin = "center"
    Rc = AXIS_CANDIDATES[axes or BOARD_AXES]
    return P @ np.linalg.inv(Rc).T                           # p_board = Rc^-1 p_cv

_DET = {}
def detect_charuco(gray, spec):
    """-> (corner_ids (n,), uv (n,2)) in the board's ChArUco corner indexing."""
    key = (spec["dictionary"], spec["squares"], spec["square_len"])
    if key not in _DET: _DET[key] = make_board(spec)
    board, dic = _DET[key]
    if hasattr(cv2.aruco, "CharucoDetector"):
        det = cv2.aruco.CharucoDetector(board)
        cc, ci, _, _ = det.detectBoard(gray)
        if cc is None or len(cc) < 4: return None, None
        return ci.ravel().astype(int), cc.reshape(-1, 2)
    mc, mi, _ = cv2.aruco.detectMarkers(gray, dic)
    if mi is None or len(mi) < 2: return None, None
    n, cc, ci = cv2.aruco.interpolateCornersCharuco(mc, mi, gray, board)
    if n is None or n < 4: return None, None
    return ci.ravel().astype(int), cc.reshape(-1, 2)

def pnp_board(ids, uv, spec, K, axes=None):
    """-> (T_cam_board, mean_reproj_px) or (None, inf)."""
    P = board_object_points(spec, axes)[ids]
    if len(P) < 4: return None, np.inf
    ok, rv, tv = cv2.solvePnP(P.astype(np.float64), uv.astype(np.float64), K, None,
                              flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return None, np.inf
    rv, tv = cv2.solvePnPRefineLM(P.astype(np.float64), uv.astype(np.float64), K, None, rv, tv)
    proj, _ = cv2.projectPoints(P, rv, tv, K, None)
    err = float(np.mean(np.linalg.norm(proj.reshape(-1, 2) - uv, axis=1)))
    return Rt(cv2.Rodrigues(rv)[0], tv.ravel()), err

print("charuco helpers ready; axis candidates:", list(AXIS_CANDIDATES))

In [ ]:
# ============================== STEP 0b: run the census ==============================
FORCE = False
CENSUS = WORK / "census_m2.npz"
SPECS  = {n: BOARDS[n] for n in BOARDS}

if FORCE or not CENSUS.exists():
    # one detector pass per distinct (dictionary, geometry); collisions resolved later by position
    uniq, seen = [], set()
    for n, b in SPECS.items():
        k = (b["dictionary"], b["squares"], b["square_len"])
        if k not in seen: seen.add(k); uniq.append((k, b))
    print("detector passes:", [k for k, _ in uniq])

    rows, t0 = [], time.time()
    for fi, (t, m) in enumerate(iter_topic(BAG, T_INFRA1, stride=CENSUS_STRIDE, limit=LIMIT_FRAMES)):
        im = img_to_np(m)
        gray = im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        for k, spec in uniq:
            ids, uv = detect_charuco(gray, spec)
            if ids is None or len(ids) < MIN_CORNERS: continue
            T_cb, err = pnp_board(ids, uv, spec, K_IR)
            if T_cb is None or err > MAX_REPROJ_PX: continue
            rows.append(dict(t=t, frame=fi, key=k[0] + str(k[1]), n=len(ids),
                             rng=float(np.linalg.norm(T_cb[:3, 3])), err=err,
                             T=T_cb, ids=ids, uv=uv))
        if fi % 200 == 0:
            print(f"  {fi:5d} frames  {len(rows):4d} sightings  {time.time()-t0:5.1f}s", flush=True)
    np.savez_compressed(CENSUS, rows=np.array(rows, dtype=object), allow_pickle=True)
    print(f"detected {len(rows)} raw sightings in {time.time()-t0:.1f}s")

RAW = list(np.load(CENSUS, allow_pickle=True)["rows"])
print(f"loaded {len(RAW)} raw sightings")
if RAW:
    import collections
    print(collections.Counter(r["key"] for r in RAW))
    print(f"range {min(r['rng'] for r in RAW):.2f}-{max(r['rng'] for r in RAW):.2f} m, "
          f"mean reproj {np.mean([r['err'] for r in RAW]):.3f} px "
          f"(survey was 0.22-0.39 px; much worse => IR projector on, try USE_COLOR)")
else:
    print("!! NO SIGHTINGS. Check topic, MIN_CORNERS, and whether mobile_2 ever faced a board.")

In [ ]:
# ============================== load VSLAM odometry ==============================
vo_t, vo_T = [], []
for t, m in iter_topic(BAG, T_VO, limit=LIMIT_FRAMES):
    vo_t.append(t); vo_T.append(odom_to_T(m))
vo_t = np.array(vo_t); vo_T = np.array(vo_T)
print(f"VSLAM odometry: {len(vo_t)} poses, {vo_t[-1]-vo_t[0]:.1f} s, "
      f"path {np.sum(np.linalg.norm(np.diff(vo_T[:,:3,3],axis=0),axis=1)):.1f} m")

In [ ]:
# ============================== resolve the ID collision, cluster sightings ==============
# anchor and anchor_b are the same physical design, so a detection only tells us "some
# DICT_4X4_50 9x7 board". Sightings are clustered by board-origin position in the VSLAM
# frame (origins are invariant to the axis convention, so this runs before the axis check), then clusters are matched to surveyed boards by their distance to an unambiguous
# board. The surveyed rs->anchor (16.59 m) and rs->anchor_b (7.10 m) differ by 9.5 m, so the
# assignment survives metres of VSLAM drift.
SIGHT = []
for r in RAW:
    spec = next(b for b in SPECS.values() if b["dictionary"] + str(b["squares"]) == r["key"])
    T_cb, err = pnp_board(r["ids"], r["uv"], spec, K_IR, axes=BOARD_AXES)
    if T_cb is None or err > MAX_REPROJ_PX: continue
    T_vo = interp_traj(vo_t, vo_T, np.array([r["t"]]))[0]
    SIGHT.append(dict(**{k: r[k] for k in ("t", "n", "rng", "err", "ids", "uv", "key")},
                      T_cb=T_cb, p_vo=(T_vo @ T_cb)[:3, 3]))

def cluster(pts, tol=0.6):
    lab = -np.ones(len(pts), int); c = 0
    for i in range(len(pts)):
        if lab[i] >= 0: continue
        m = np.linalg.norm(pts - pts[i], axis=1) < tol
        lab[m] = c; c += 1
    return lab, c

CLUS = {}
for key in sorted({s["key"] for s in SIGHT}):
    idx = [i for i, s in enumerate(SIGHT) if s["key"] == key]
    P = np.array([SIGHT[i]["p_vo"] for i in idx])
    lab, nc = cluster(P)
    for c in range(nc):
        sel = [idx[k] for k in np.where(lab == c)[0]]
        CLUS[f"{key}#{c}"] = dict(key=key, idx=sel,
                                  p_vo=np.median([SIGHT[i]["p_vo"] for i in sel], axis=0),
                                  t0=min(SIGHT[i]["t"] for i in sel),
                                  t1=max(SIGHT[i]["t"] for i in sel))
print(f"{len(SIGHT)} sightings -> {len(CLUS)} spatial clusters")
for cid, c in CLUS.items():
    print(f"  {cid:28s} n={len(c['idx']):4d}  t=[{c['t0']-vo_t[0]:6.1f},{c['t1']-vo_t[0]:6.1f}]s"
          f"  p_vo={np.round(c['p_vo'],2)}")

# --- assign clusters to surveyed boards --------------------------------
unamb = {n: b for n, b in BOARDS.items()
         if not any(n in v for v in AMBIG.values())}
ref_cid = next((cid for cid, c in CLUS.items()
                if any(c["key"] == b["dictionary"] + str(b["squares"]) for b in unamb.values())), None)
ASSIGN = {}
if ref_cid is None:
    print("\n!! no unambiguous board seen - assignment cannot be resolved automatically.")
    print("   Set ASSIGN by hand, e.g. ASSIGN = {'DICT_4X4_50(9, 7)#0': 'anchor', ...}")
else:
    ref_name = next(n for n, b in unamb.items()
                    if b["dictionary"] + str(b["squares"]) == CLUS[ref_cid]["key"])
    ASSIGN[ref_cid] = ref_name
    p_ref = CLUS[ref_cid]["p_vo"]
    print(f"\nreference cluster {ref_cid} = '{ref_name}'")
    for cid, c in CLUS.items():
        if cid == ref_cid: continue
        d_meas = float(np.linalg.norm(c["p_vo"] - p_ref))
        cands = {n: float(np.linalg.norm(b["T_map_board"][:3, 3] -
                                         BOARDS[ref_name]["T_map_board"][:3, 3]))
                 for n, b in BOARDS.items()
                 if n != ref_name and b["dictionary"] + str(b["squares"]) == c["key"]}
        if not cands: continue
        best = min(cands, key=lambda n: abs(cands[n] - d_meas))
        ASSIGN[cid] = best
        others = " ".join(f"{n}={cands[n]:.2f}" for n in cands)
        print(f"  {cid:28s} d_meas={d_meas:6.2f} m -> '{best}'   (surveyed: {others})")
        if len(cands) > 1:
            gap = sorted(abs(cands[n] - d_meas) for n in cands)
            print(f"     margin {gap[1]-gap[0]:.2f} m  {'OK' if gap[1]-gap[0] > 1.0 else '<< AMBIGUOUS'}")

for cid, name in ASSIGN.items():
    for i in CLUS[cid]["idx"]: SIGHT[i]["board"] = name
SIGHT = [s for s in SIGHT if "board" in s]
print(f"\n{len(SIGHT)} assigned sightings across {len(set(s['board'] for s in SIGHT))} boards")

In [ ]:
# ============================== board-axis convention check ==============================
# PnP under ANY rigid axis convention reproduces the same corners and the same board origin -
# the convention only redefines the board's ORIENTATION frame. So it cannot be picked from
# mobile_2's detections alone; it must match what the SURVEY used when it stored each qxyzw.
# The discriminating signal: the relative rotation between two boards, survey-stored vs
# measured through VSLAM. A wrong convention is off by ~90-180 deg; VSLAM rotation drift over
# the between-board stretch is ~1 deg. Needs sightings of >= 2 distinct boards.
def rel_rot_score(axes):
    Rm = {}
    for s in SIGHT:
        T_cb, err = pnp_board(s["ids"], s["uv"], BOARDS[s["board"]], K_IR, axes=axes)
        if T_cb is None or err > MAX_REPROJ_PX: continue
        T_vo = interp_traj(vo_t, vo_T, np.array([s["t"]]))[0]
        Rm.setdefault(s["board"], []).append((T_vo @ T_cb)[:3, :3])
    names = sorted(Rm)
    if len(names) < 2: return None
    worst = 0.0
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            R_meas = Rm[a][len(Rm[a]) // 2].T @ Rm[b][len(Rm[b]) // 2]
            R_srv = BOARDS[a]["T_map_board"][:3, :3].T @ BOARDS[b]["T_map_board"][:3, :3]
            worst = max(worst, math.degrees(np.linalg.norm(
                Rot.from_matrix(R_srv.T @ R_meas).as_rotvec())))
    return worst                      # deg; the right convention is the ~drift-sized one

scores = {a: rel_rot_score(a) for a in AXIS_CANDIDATES}
if all(v is None for v in scores.values()):
    print("!! fewer than 2 distinct boards sighted - the convention cannot be checked from "
          "this bag.\n   Set BOARD_AXES by hand from the survey tool's own definition "
          "(its config says board_axes='ros').")
else:
    for a, v in sorted(scores.items(), key=lambda kv: (kv[1] is None, kv[1])):
        print(f"  {a:9s} relative-rotation disagreement "
              + ("   n/a" if v is None else f"{v:8.2f} deg"))
    BOARD_AXES = min((a for a in scores if scores[a] is not None), key=lambda a: scores[a])
    best = scores[BOARD_AXES]
    print(f"\n-> BOARD_AXES = '{BOARD_AXES}'  ({best:.2f} deg)")
    if best > 10:
        print("!! even the best candidate disagrees by >10 deg - none of the enumerated "
              "conventions matches the survey. Read the survey tool's board-frame code and "
              "add its rotation to AXIS_CANDIDATES.")
    # re-solve every stored sighting pose under the chosen convention
    for s in SIGHT:
        T_cb, err = pnp_board(s["ids"], s["uv"], BOARDS[s["board"]], K_IR, axes=BOARD_AXES)
        if T_cb is not None: s["T_cb"], s["err"] = T_cb, err

In [ ]:
# ============================== CENSUS GATE ==============================
t_rel = np.array([s["t"] for s in SIGHT]) - vo_t[0]
bnames = sorted({s["board"] for s in SIGHT})
dur = vo_t[-1] - vo_t[0]

# a "window" = contiguous sightings of one board with < 2 s gaps
WINDOWS = []
for b in bnames:
    tb = np.sort(t_rel[[i for i, s in enumerate(SIGHT) if s["board"] == b]])
    if not len(tb): continue
    br = np.where(np.diff(tb) > 2.0)[0]
    for a, z in zip(np.r_[0, br + 1], np.r_[br, len(tb) - 1]):
        WINDOWS.append((b, tb[a], tb[z], z - a + 1))

print(f"trajectory {dur:.1f} s   sighting duty cycle "
      f"{100*len(SIGHT)/max(1,len(vo_t)/CENSUS_STRIDE):.1f}%")
print(f"\n{'board':11s} {'t_start':>8s} {'t_end':>8s} {'n':>5s}")
for b, a, z, n in sorted(WINDOWS, key=lambda w: w[1]):
    print(f"{b:11s} {a:8.1f} {z:8.1f} {n:5d}")
gaps = np.diff(np.r_[0, sorted(w[1] for w in WINDOWS), dur])
print(f"\n{len(WINDOWS)} windows; longest board-free stretch {gaps.max():.1f} s "
      f"({gaps.max()*0.8:.1f} m at 0.8 m/s)")
print("GATE:", "PASS - three-arm ablation is meaningful" if len(WINDOWS) >= 4 else
      "FAIL - reframe as anchored vs unanchored VSLAM, not a fusion ablation")

fig, ax = plt.subplots(2, 1, figsize=(11, 6), height_ratios=[1, 2])
for i, b in enumerate(bnames):
    m = [j for j, s in enumerate(SIGHT) if s["board"] == b]
    ax[0].scatter(t_rel[m], np.full(len(m), i), s=8, label=b)
    ax[1].scatter(t_rel[m], [SIGHT[j]["rng"] for j in m], s=8, label=b)
ax[0].set_yticks(range(len(bnames))); ax[0].set_yticklabels(bnames)
ax[0].set_xlim(0, dur); ax[0].set_title("board sightings over the trajectory"); ax[0].grid(alpha=.3)
ax[1].set_xlim(0, dur); ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("range [m]")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Step 0c - are the boards still where the survey says?  **(GATE)**

The survey is stamped `1787894041`; this bag starts `1787899802` - **96 minutes later**. The
surveyed board poses only transfer if nothing was bumped in between.

The check below is extrinsic-free: measure each inter-board baseline *inside this bag* by
composing two sightings through the VSLAM trajectory, and compare to the survey. It is a
**joint** test of board stability and VSLAM metric scale, so read it that way - agreement
validates both; disagreement tells you something is wrong without saying which. That is still
worth having before you build three estimators on top of these boards.

The stronger, single-hypothesis test is to redetect the boards from `mobile_1`'s ZED in this
same bag, since `mobile_1` carries the Ouster and so has a GLIM pose in the map frame. It needs
the LiDAR->ZED extrinsic, which is not in this bag's 3-message `/tf_static`; do it separately
if this cell comes back disagreeing.

In [ ]:
# ============================== STEP 0c: board stability / VO scale ==============
print(f"{'pair':26s} {'surveyed':>9s} {'measured':>9s} {'diff':>8s} {'n':>5s}")
rows = []
for i in range(len(bnames)):
    for j in range(i + 1, len(bnames)):
        a, b = bnames[i], bnames[j]
        d_srv = float(np.linalg.norm(BOARDS[a]["T_map_board"][:3, 3] -
                                     BOARDS[b]["T_map_board"][:3, 3]))
        pa = np.array([s["p_vo"] for s in SIGHT if s["board"] == a])
        pb = np.array([s["p_vo"] for s in SIGHT if s["board"] == b])
        d_msr = float(np.linalg.norm(np.median(pa, 0) - np.median(pb, 0)))
        rows.append((d_srv, d_msr))
        print(f"{a+' <-> '+b:26s} {d_srv:8.3f}m {d_msr:8.3f}m {1000*(d_msr-d_srv):+7.0f}mm"
              f" {len(pa)+len(pb):5d}")
if len(rows) >= 2:
    s = np.polyfit([r[0] for r in rows], [r[1] for r in rows], 1)[0]
    print(f"\nimplied VSLAM scale {s:.5f}  ({(s-1)*1e6:+.0f} ppm)")
    print("  interpret: a consistent scale != 1 across pairs => VSLAM stereo scale error "
          "(estimable, ESTIMATE_VO_SCALE=True handles it).")
    print("  a single pair disagreeing while others match => that board moved.")
elif rows:
    print("\nonly one pair - cannot separate 'board moved' from 'VSLAM scale'. "
          "Treat a large diff as a warning, not a diagnosis.")

In [ ]:
# ============================== save for the next notebook ==============================
out = WORK / "step0_sightings.npz"
np.savez_compressed(
    out,
    sightings=np.array(SIGHT, dtype=object),
    board_axes=BOARD_AXES,
    assign=np.array(list(ASSIGN.items()), dtype=object),
    boards=np.array([(n, b["T_map_board"], b["sigma_t"], b["sigma_r"])
                     for n, b in BOARDS.items()], dtype=object),
    allow_pickle=True)
print(f"wrote {out}: {len(SIGHT)} assigned sightings, BOARD_AXES='{BOARD_AXES}'")
print("""
Decision record - fill in before moving on:
  0a  weakest board sigma used ........ (anchor_b 12.9 mm / 2.64 deg unless re-surveyed)
  0b  gate .............................. PASS (>=4 windows) / FAIL (reframe, no fusion ablation)
  0c  baselines agree within ........... mm  -> boards trusted? VSLAM scale sane?
Next notebook: depth -> clouds, scan-to-map registration, and the A/B/C pose graph,
consuming this file.""")